In [1]:
import pandas as pd

sequences = pd.read_csv("/content/activation_state_sequences.csv")
tokens = pd.read_csv("/content/token_state_assignments.csv")

print("SEQUENCES")
print("Shape:", sequences.shape)
print("Columns:", sequences.columns.tolist())
display(sequences.head(3))

print("\nTOKEN ASSIGNMENTS")
print("Shape:", tokens.shape)
print("Columns:", tokens.columns.tolist())
display(tokens.head(3))

SEQUENCES
Shape: (150, 5)
Columns: ['Prompt_ID', 'Response_Label', 'Layer_ID', 'State_Sequence', 'Sequence_Length']


,Prompt_ID,Response_Label,Layer_ID,State_Sequence,Sequence_Length
0,19,0,10,"[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, ...",64
1,19,0,18,"[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, ...",64
2,19,0,26,"[8, 8, 8, 0, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, ...",64



TOKEN ASSIGNMENTS
Shape: (7659, 5)
Columns: ['Prompt_ID', 'Response_Label', 'Layer_ID', 'Token_Step', 'State_ID']


,Prompt_ID,Response_Label,Layer_ID,Token_Step,State_ID
0,140,1,10,0,3
1,140,1,10,1,3
2,140,1,10,2,3


In [2]:
from sklearn.model_selection import train_test_split
import json
import ast

# Convert State_Sequence text into Python lists
def parse_sequence(x):
    if isinstance(x, list):
        return x
    try:
        return json.loads(x)
    except:
        return ast.literal_eval(x)

sequences["State_Sequence"] = sequences["State_Sequence"].apply(parse_sequence)

# One row per prompt for splitting
prompt_labels = (
    sequences[["Prompt_ID", "Response_Label"]]
    .drop_duplicates()
)

train_ids, test_ids = train_test_split(
    prompt_labels["Prompt_ID"],
    test_size=0.20,
    random_state=42,
    stratify=prompt_labels["Response_Label"]
)

train_ids = set(train_ids)
test_ids = set(test_ids)

train_df = sequences[sequences["Prompt_ID"].isin(train_ids)].copy()
test_df = sequences[sequences["Prompt_ID"].isin(test_ids)].copy()

print("Train prompts:", len(train_ids))
print("Test prompts:", len(test_ids))

print("\nTrain labels:")
print(
    prompt_labels[prompt_labels["Prompt_ID"].isin(train_ids)]
    ["Response_Label"].value_counts().sort_index()
)

print("\nTest labels:")
print(
    prompt_labels[prompt_labels["Prompt_ID"].isin(test_ids)]
    ["Response_Label"].value_counts().sort_index()
)

print("\nTrain sequence rows:", len(train_df))
print("Test sequence rows:", len(test_df))

print("\nPrompt overlap:", len(train_ids.intersection(test_ids)))

Train prompts: 40
Test prompts: 10

Train labels:
Response_Label
0    20
1    20
Name: count, dtype: int64

Test labels:
Response_Label
0    5
1    5
Name: count, dtype: int64

Train sequence rows: 120
Test sequence rows: 30

Prompt overlap: 0


In [4]:
!pip install prefixspan

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Created wheel for prefixspan: filename=prefixspan-0.5.2-py3-none-any.whl size=11314 sha256=4ef8ce1b97a9c2ca5e9073c073a522968f5351fadaf6a2aaaaa9eefe6d7eac44
  Stored in directory: /root/.cache/pip/wheels/d0/4a/de/192072146c56cc4ee4d9669c4d0ffdeabbc8dfc4759f397d4e
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13781 sha256=c1190ac8be4e06aa07dec64d3aabf40de36093b8dc3daafed83e02d94a896869
  Stored in directory: /root/.cache/pip/wheels/0b/1d/03/175286677fb5a1341cc3e4755bf8ec0ed08f3329afd67446b0
  Created wheel for extratools: filename=extratools-0.8.2.1-py3-none-any.whl size=28974 sha256=9c0fbc450a67ee1594b3122dc8c465a8bd311a5d3ad93d6774f4813e5c53ac56
  Stored in directory: /root/.cache/pip/wheels/1f/b7/10/95b187b30d744aaba41c25980d5b9998bf0c90580ea7c87d12
Successfully built prefixspan docopt extratools


In [5]:
from prefixspan import PrefixSpan

# Layer 26 training prompts
train_l26 = train_df[train_df["Layer_ID"] == 26].copy()

truthful_seqs = train_l26[train_l26["Response_Label"] == 0]["State_Sequence"].tolist()
hallucinated_seqs = train_l26[train_l26["Response_Label"] == 1]["State_Sequence"].tolist()

# Maximum pattern length 5; minimum support 4
ps_truth = PrefixSpan(truthful_seqs)
ps_truth.maxlen = 5
truth_patterns = ps_truth.frequent(4)

ps_hall = PrefixSpan(hallucinated_seqs)
ps_hall.maxlen = 5
hall_patterns = ps_hall.frequent(4)

candidate_patterns = set(
    tuple(pattern) for support, pattern in truth_patterns
)
candidate_patterns.update(
    tuple(pattern) for support, pattern in hall_patterns
)

print("Train Layer 26 sequences:", len(train_l26))
print("Truthful patterns mined:", len(truth_patterns))
print("Hallucinated patterns mined:", len(hall_patterns))
print("Unique candidate patterns:", len(candidate_patterns))

Train Layer 26 sequences: 40
Truthful patterns mined: 1050
Hallucinated patterns mined: 1084
Unique candidate patterns: 1406


In [6]:
def contains_subsequence(sequence, pattern):
    j = 0
    for state in sequence:
        if j < len(pattern) and state == pattern[j]:
            j += 1
        if j == len(pattern):
            return True
    return False

train_records = train_l26[["Prompt_ID", "Response_Label", "State_Sequence"]].to_dict("records")

pattern_rows = []

for pattern in candidate_patterns:
    truthful_support = sum(
        r["Response_Label"] == 0 and contains_subsequence(r["State_Sequence"], pattern)
        for r in train_records
    )
    hallucinated_support = sum(
        r["Response_Label"] == 1 and contains_subsequence(r["State_Sequence"], pattern)
        for r in train_records
    )

    total = truthful_support + hallucinated_support
    if total == 0:
        continue

    confidence = hallucinated_support / total
    lift = confidence / 0.5
    contrast = (hallucinated_support / 20) - (truthful_support / 20)

    pattern_rows.append({
        "Pattern": list(pattern),
        "Pattern_Length": len(pattern),
        "Truthful_Support": truthful_support,
        "Hallucinated_Support": hallucinated_support,
        "Confidence": confidence,
        "Lift": lift,
        "Contrast": contrast
    })

train_pattern_metrics = pd.DataFrame(pattern_rows)

train_pattern_library = train_pattern_metrics[
    (train_pattern_metrics["Hallucinated_Support"] >= 4) &
    (train_pattern_metrics["Confidence"] >= 0.70) &
    (train_pattern_metrics["Lift"] >= 1.20) &
    (train_pattern_metrics["Contrast"] >= 0.20)
].copy()

print("Candidate patterns evaluated:", len(train_pattern_metrics))
print("Train-derived hallucination-associated patterns:", len(train_pattern_library))
train_pattern_library.head()

Candidate patterns evaluated: 1406
Train-derived hallucination-associated patterns: 142


,Pattern,Pattern_Length,Truthful_Support,Hallucinated_Support,Confidence,Lift,Contrast
11,"[2, 0, 4, 9, 9]",5,0,4,1.000000,2.000000,0.20
15,"[4, 4, 3]",3,3,10,0.769231,1.538462,0.35
20,"[4, 0, 3, 4, 4]",5,3,8,0.727273,1.454545,0.25
34,"[4, 4, 3, 4, 9]",5,3,9,0.750000,1.500000,0.30
35,"[3, 4, 4, 3, 4]",5,1,5,0.833333,1.666667,0.20


In [7]:
test_l26 = test_df[test_df["Layer_ID"] == 26].copy()

library = [tuple(p) for p in train_pattern_library["Pattern"].tolist()]

def earliest_warning_fraction(sequence, patterns):
    if not patterns:
        return None

    for end in range(1, len(sequence) + 1):
        prefix = sequence[:end]
        if any(contains_subsequence(prefix, p) for p in patterns):
            return end / len(sequence)

    return None

test_results = []

for _, row in test_l26.iterrows():
    fraction = earliest_warning_fraction(row["State_Sequence"], library)
    warning = fraction is not None

    test_results.append({
        "Prompt_ID": row["Prompt_ID"],
        "Actual_Label": row["Response_Label"],
        "Warning": int(warning),
        "Warning_Fraction": fraction
    })

test_results = pd.DataFrame(test_results)

from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

y_true = test_results["Actual_Label"]
y_pred = test_results["Warning"]

tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

print("TEST RESULTS — 10 held-out prompts")
print("TP:", tp, "FP:", fp, "FN:", fn, "TN:", tn)
print("Accuracy:", round(accuracy_score(y_true, y_pred), 3))
print("Precision:", round(precision_score(y_true, y_pred, zero_division=0), 3))
print("Recall:", round(recall_score(y_true, y_pred, zero_division=0), 3))
print("F1:", round(f1_score(y_true, y_pred, zero_division=0), 3))
print("Warnings:", int(y_pred.sum()), "/", len(y_pred))
print("Average warning fraction:",
      round(test_results["Warning_Fraction"].dropna().mean(), 3)
      if test_results["Warning_Fraction"].notna().any() else "No warnings")

display(test_results)

TEST RESULTS — 10 held-out prompts
TP: 4 FP: 2 FN: 1 TN: 3
Accuracy: 0.7
Precision: 0.667
Recall: 0.8
F1: 0.727
Warnings: 6 / 10
Average warning fraction: 0.61


,Prompt_ID,Actual_Label,Warning,Warning_Fraction
0,164,0,1,0.981818
1,183,0,0,NaN
2,310,1,0,NaN
3,324,0,0,NaN
4,347,1,1,0.625000
5,421,1,1,0.750000
6,488,0,1,0.941176
7,530,0,0,NaN
8,637,1,1,0.150000
9,789,1,1,0.212121


In [8]:
import json
import os
import numpy as np

os.makedirs("results/final_evaluation", exist_ok=True)

summary = {
    "evaluation_type": "Prompt-level held-out evaluation",
    "train_prompts": 40,
    "test_prompts": 10,
    "train_test_prompt_overlap": 0,
    "test_confusion_matrix": {
        "TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn)
    },
    "test_metrics": {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0))
    },
    "warnings": int(y_pred.sum()),
    "average_warning_fraction": float(
        test_results["Warning_Fraction"].dropna().mean()
    ) if test_results["Warning_Fraction"].notna().any() else None,
    "limitations": [
        "Only 10 held-out test prompts were used.",
        "Scaler, PCA, and KMeans were fitted using all 50 prompts before this split.",
        "Therefore, this is not a fully leakage-free end-to-end evaluation.",
        "Results are preliminary and should not be generalized."
    ]
}

with open("results/final_evaluation/evaluation_summary.json", "w") as f:
    json.dump(summary, f, indent=4)

print("Saved evaluation_summary.json")

Saved evaluation_summary.json


In [9]:
test_results.to_csv(
    "results/final_evaluation/held_out_test_results.csv",
    index=False
)

train_pattern_library.to_csv(
    "results/final_evaluation/train_pattern_library.csv",
    index=False
)

train_pattern_metrics.to_csv(
    "results/final_evaluation/train_pattern_metrics.csv",
    index=False
)

print("Saved all 3 CSV files.")

Saved all 3 CSV files.


In [10]:
report_text = f"""
## Held-out Evaluation

The dataset was split at the prompt level into 40 training prompts and
10 test prompts. Both sets were balanced across the two response labels,
and there was no prompt overlap.

Sequential patterns were mined using only the training prompts from
Layer 26. The resulting pattern library was evaluated on the 10 held-out
test prompts.

Test results:
- True Positives: {tp}
- False Positives: {fp}
- False Negatives: {fn}
- True Negatives: {tn}
- Accuracy: {accuracy_score(y_true, y_pred):.3f}
- Precision: {precision_score(y_true, y_pred, zero_division=0):.3f}
- Recall: {recall_score(y_true, y_pred, zero_division=0):.3f}
- F1-score: {f1_score(y_true, y_pred, zero_division=0):.3f}

The average warning fraction was {test_results["Warning_Fraction"].dropna().mean():.3f}.
This means warnings were raised, on average, after this fraction of the
generated activation-state sequence had been observed.

Limitations:
The test set contains only 10 prompts, so the metrics are preliminary.
Also, scaling, PCA, and KMeans discretization were fitted using all
50 prompts before the train-test split. Therefore, this is a held-out
evaluation of pattern mining and warning matching conditional on the
existing discretization, not a fully leakage-free end-to-end evaluation.
"""

with open("results/final_evaluation/report_summary.txt", "w") as f:
    f.write(report_text)

print(report_text)


## Held-out Evaluation

The dataset was split at the prompt level into 40 training prompts and
10 test prompts. Both sets were balanced across the two response labels,
and there was no prompt overlap.

Sequential patterns were mined using only the training prompts from
Layer 26. The resulting pattern library was evaluated on the 10 held-out
test prompts.

Test results:
- True Positives: 4
- False Positives: 2
- False Negatives: 1
- True Negatives: 3
- Accuracy: 0.700
- Precision: 0.667
- Recall: 0.800
- F1-score: 0.727

The average warning fraction was 0.610.
This means warnings were raised, on average, after this fraction of the
generated activation-state sequence had been observed.

Limitations:
The test set contains only 10 prompts, so the metrics are preliminary.
Also, scaling, PCA, and KMeans discretization were fitted using all
50 prompts before the train-test split. Therefore, this is a held-out
evaluation of pattern mining and warning matching conditional on the
existing discr

In [11]:
import shutil
from google.colab import files

shutil.make_archive(
    "final_evaluation_outputs",
    "zip",
    "results/final_evaluation"
)

files.download("final_evaluation_outputs.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>